In [8]:
import nbimporter
from Active_Learning_on_3_datasets_1st_presentation.ipynb import *

ModuleNotFoundError: No module named 'Active_Learning_on_3_datasets_1st_presentation'

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import struct
import warnings

from typing import Tuple, Literal, Optional, List, Dict
# Importation des modules nécessaires de scikit-learn
from scipy.stats import entropy
from scipy.spatial.distance import euclidean
from scipy.spatial.distance import cityblock


from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,f1_score, average_precision_score
from sklearn.metrics import precision_recall_curve, auc, average_precision_score

from sklearn.base import clone
from sklearn.base import ClassifierMixin

# Classificateurs de scikit-learn
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.linear_model import (
    LogisticRegression,
    RidgeClassifier,
    ElasticNetCV
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier,NearestNeighbors
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE


# LightGBM
import lightgbm

# Configuration du logging
import logging
# Ignorer les warnings
warnings.filterwarnings("ignore")


In [4]:
df1=pd.read_csv("all_reviews_features 2.csv")
df2=pd.read_csv("reviews_since_february_with_features 1.csv")
df3=pd.read_csv("reviews_test_nb_bets 2.csv")
df=pd.concat([df1,df2,df3])

# Charger les datasets
df1 = pd.read_csv("all_reviews_features 2.csv")
df2 = pd.read_csv("reviews_since_february_with_features 1.csv")
df3 = pd.read_csv("reviews_test_nb_bets 2.csv")

# Combinaison des datasets
df = pd.concat([df1, df2, df3])

# Définir les caractéristiques
FEATURES = ['SINGLE_PROPORTION_30D', 'MARKET_MARGIN_30D', 'TURNOVER_PER_BET_30D', 'DEPOSIT_MAX_30D', 
            'SHIELD_REJECTION_30D', 'LIVE_PROPORTION_30D', 'GGR_PER_BET_30D', 'MARGIN_30D', 
            'MARGIN_PER_BET_30D', 'CLOSING_LINE_VALUE_30D', 'MAX_STAKE_RATIO_30D', 'MARKET_TYPE_SCORE_30D', 
            'BET_SCORE_30D', 'TIME_BEFORE_EVENT_30D', 'LATE_BET_ACTION_COUNT_30D', 
            'LATE_BET_STOLEN_AMOUNT_EURO_30D', 'SINGLE_PROPORTION_10D', 'MARKET_MARGIN_10D', 
            'TURNOVER_PER_BET_10D', 'DEPOSIT_MAX_10D', 'SHIELD_REJECTION_10D', 'LIVE_PROPORTION_10D', 
            'GGR_PER_BET_10D', 'MARGIN_10D', 'MARGIN_PER_BET_10D', 'CLOSING_LINE_VALUE_10D', 
            'MAX_STAKE_RATIO_10D', 'MARKET_TYPE_SCORE_10D', 'BET_SCORE_10D', 'TIME_BEFORE_EVENT_10D', 
            'LATE_BET_ACTION_COUNT_10D', 'LATE_BET_STOLEN_AMOUNT_EURO_10D', 'SELECTION_DIVERSITY_LAST_0_DAYS', 
            'SHARP_SHARE_LAST_0_DAYS', 'YELLOW_BREAKOUT_LAST_0_DAYS', 'RISKY_RING_RATIO_LAST_0_DAYS', 
            'SINGLE_PROPORTION_365D', 'MARKET_MARGIN_365D', 'TURNOVER_PER_BET_365D', 'DEPOSIT_MAX_365D', 
            'SHIELD_REJECTION_365D', 'LIVE_PROPORTION_365D', 'GGR_PER_BET_365D', 'MARGIN_PER_BET_365D', 
            'MARGIN_365D', 'CLOSING_LINE_VALUE_365D', 'MARKET_TYPE_SCORE_365D', 'BET_SCORE_365D', 
            'SHARP_SHARE_365D', 'TIME_BEFORE_EVENT_365D', 'MAX_STAKE_RATIO_365D', 'SINGLE_PROPORTION_180D', 
            'MARKET_MARGIN_180D', 'TURNOVER_PER_BET_180D', 'DEPOSIT_MAX_180D', 'SHIELD_REJECTION_180D', 
            'LIVE_PROPORTION_180D', 'GGR_PER_BET_180D', 'MARGIN_PER_BET_180D', 'MARGIN_180D', 
            'CLOSING_LINE_VALUE_180D', 'MARKET_TYPE_SCORE_180D', 'BET_SCORE_180D', 'SHARP_SHARE_180D', 
            'TIME_BEFORE_EVENT_180D', 'MAX_STAKE_RATIO_180D', 'SINGLE_PROPORTION_90D', 'MARKET_MARGIN_90D', 
            'TURNOVER_PER_BET_90D', 'DEPOSIT_MAX_90D', 'SHIELD_REJECTION_90D', 'LIVE_PROPORTION_90D', 
            'GGR_PER_BET_90D', 'MARGIN_PER_BET_90D', 'MARGIN_90D', 'CLOSING_LINE_VALUE_90D', 
            'MARKET_TYPE_SCORE_90D', 'BET_SCORE_90D', 'SHARP_SHARE_90D', 'TIME_BEFORE_EVENT_90D', 
            'MAX_STAKE_RATIO_90D', 'LATE_BET_ACTION_COUNT_90D', 'LATE_BET_STOLEN_AMOUNT_EURO_90D', 
            'SHARP_SHARE_30D', 'SHARP_SHARE_10D', 'has_previous', 'NB_BET_LAST_10_DAYS', 
            'NB_BET_LAST_30_DAYS', 'NB_BET_LAST_90_DAYS', 'NB_BET_LAST_180_DAYS', 'NB_BET_LAST_365_DAYS', 
            'TURNOVER_LAST_10_DAYS', 'TURNOVER_LAST_30_DAYS', 'TURNOVER_LAST_90_DAYS', 
            'TURNOVER_LAST_180_DAYS', 'TURNOVER_LAST_365_DAYS', 'DEPOSIT_LAST_10_DAYS', 
            'DEPOSIT_LAST_30_DAYS', 'DEPOSIT_LAST_90_DAYS', 'DEPOSIT_LAST_180_DAYS', 'DEPOSIT_LAST_365_DAYS', 
            'GGR_LAST_10_DAYS', 'GGR_LAST_30_DAYS', 'GGR_LAST_90_DAYS', 'GGR_LAST_180_DAYS', 
            'GGR_LAST_365_DAYS', 'ORIGINAL_FLAG']

# Mapper les colonnes 'CURRENT_FLAG' et 'ORIGINAL_FLAG' avec color_values
color_values = {
    "Yellow": 2,
    "Brown": 2,
    "Red": 2,
    "Black": 2,
    "Grey": 2,
    "Pink": 2,
    "No flag": 0,
    "Blue": 1,
    "Bluish": 1,
    "Purple": 1,
    "Green": 0,
    "Salmon": 1,
}

df['label'] = df['CURRENT_FLAG'].map(color_values).fillna(0)  # Remplacement et gestion des valeurs manquantes
df['ORIGINAL_FLAG'] = df['ORIGINAL_FLAG'].map(color_values).fillna(0)
"""
df['label'] = df['label'].replace(1, 0)  # Correction spécifique si nécessaire
df['label'] = df['label'].replace(2, 1)  # pour adapter à AUC

df['ORIGINAL_FLAG'] = df['ORIGINAL_FLAG'].replace(1, 0)  # Correction spécifique si nécessaire
df['ORIGINAL_FLAG'] = df['ORIGINAL_FLAG'].replace(2, 1)  # pour adapter à AUC
print(df.columns.get_loc("ORIGINAL_FLAG"),len(FEATURES),FEATURES.index("ORIGINAL_FLAG")
)"
"""
# Extraire les features et labels
X_PRS = df[FEATURES]
y_PRS = df["label"]
print(y_PRS.value_counts())
# Conversion en numpy arrays
X_PRS = X_PRS.to_numpy()
y_PRS = y_PRS.to_numpy()




label
0.0    15542
2.0     3877
1.0     2831
Name: count, dtype: int64


In [9]:
def split_dataset(X, y, labeled_ratio, test_ratio):
    """
    Divise un dataset en train, test et pool en garantissant que chaque classe soit présente dans le set d'entraînement.
    """
    # Séparer l’ensemble de test
    X_train_pool, X_test, y_train_pool, y_test = train_test_split(
        X, y, test_size=test_ratio, random_state=42, stratify=y
    )

    # Taille totale de l’ensemble train + pool
    train_pool_size = X_train_pool.shape[0]

    # Nombre d'échantillons à labelliser
    nb_labeled = int(labeled_ratio * train_pool_size)

    # Diviser train_pool en labeled et pool
    X_train, X_pool, y_train, y_pool = train_test_split(
        X_train_pool, y_train_pool, train_size=nb_labeled, random_state=42, stratify=y_train_pool
    )

    return X_train, X_pool, y_train, y_pool, X_test, y_test

In [15]:

labeled_ratio=0.01 # Nombre d'échantillons labellisés initialement
test_ratio = 0.2  # Proportion du dataset réservée au test
n_iterations = 200 # Nombre d'itérations d'Active Learning
nb_per_review=5 # Nombre d'échantillons à interroger à chaque itération
#methods=["random", "least_confident", "margin", "entropy","hybrid","qbc-variance","qbc-entropy","qbc-KL"]
methods=["random", "least_confident", "margin", "entropy"]
model_class=lambda : RandomForestClassifier()
# Créer un comité avec 3 modèles différents
# Liste de modèles pour le comité
models = [
    clone(RandomForestClassifier()),  # Random Forest
    clone(LogisticRegression(max_iter=1000)), # Régression Logistique
    clone(SVC(probability=True)),      # Régression Ridge pour classificati)           # Analyse discriminante quadratique
]
#datasets={"MNIST":(X_MNIST,y_MNIST),"PRS":(X_PRS,y_PRS),"Foot":(X_foot,y_foot)}
X,y=X_PRS,y_PRS
METRICS = {
    "MNIST": f1_score,
    "Foot": f1_score,
    "PRS": "PR AUC"  # AUC PRC
}

#  Dictionnaire pour stocker les indices sélectionnés pour chaque méthode
selected_indices_per_method = {}

#itération 1
X_train, X_pool, y_train, y_pool, X_test, y_test = split_dataset(X, y, labeled_ratio, test_ratio)

# Initialiser le classificateur
model = model_class()
# Entraîner le modèle sur les données labellisées
model.fit(X_train, y_train)
# Prédire sur le pool
y_pool_pred = model.predict_proba(X_pool)
for method in methods:
    

    if method == "random":
        query_indices = np.random.choice(len(X_pool), nb_per_review, replace=False)

    elif method == "least_confident":
        confidence = np.max(y_pool_pred, axis=1)
        query_indices = np.argsort(confidence)[:nb_per_review]

    elif method == "margin":
        if y_pool_pred.shape[1] == 2:  # binaire
            margin = np.abs(y_pool_pred[:, 0] - y_pool_pred[:, 1])
        else:
            sorted_probs = np.sort(y_pool_pred, axis=1)
            margin = sorted_probs[:, -1] - sorted_probs[:, -2]
        query_indices = np.argsort(margin)[:nb_per_review]

    elif method == "entropy":
        entropies = entropy(y_pool_pred.T)  # Entropy attend (classes, samples)
        query_indices = np.argsort(entropies)[-nb_per_review:]

    selected_indices_per_method[method] = query_indices

print("Indices sélectionnés pour chaque méthode :")
for method, indices in selected_indices_per_method.items():
    print(f"{method}: {indices}")
# Visualisation des indices sélectionnés    

Indices sélectionnés pour chaque méthode :
random: [ 5062  5824  4135  7411 12166]
least_confident: [ 3261 13249 15755 17132 17434]
margin: [12031  3348 14322  1375 13249]
entropy: [ 1984  3929  4341  9028 15805]


In [ ]:
def run_active_learning_experiment_datasets(datasets: dict, METRICS: dict,labeled_ratio: float, test_ratio: float, 
                                            n_iterations: int, batch_ratio: float, methods: list, 
                                            model_class: type, models: list) -> dict:
    """
    Exécute une expérimentation d'Active Learning sur plusieurs datasets avec différentes méthodes de sélection d'incertitude.
    
    Paramètres:
        datasets (dict): Un dictionnaire contenant les datasets sous la forme {nom_dataset: (X, y)},
                         où X est un np.ndarray (features) et y est un np.ndarray (labels).
        labeled_ratio (float): Proportion initiale d'échantillons labellisés dans l'ensemble d'entraînement.
        test_ratio (float): Proportion d'échantillons dédiés à l'ensemble de test.
        n_iterations (int): Nombre d'itérations d'Active Learning.
        batch_ratio (float): Proportion d'échantillons ajoutés à chaque itération (par rapport à la taille totale du training dataset).
        methods (list): Liste des méthodes de sélection d'incertitude à tester.
        model_class (type): Classe du modèle de classification utilisé (doit être compatible avec `fit` et `predict`).
        models (list): Liste des modèles pour les méthodes basées sur un comité (ex: Query By Committee).

    Retourne:
        dict: Un dictionnaire contenant les précisions finales pour chaque dataset et chaque méthode d'Active Learning.
    """
    
    # Initialisation des dictionnaires pour stocker les résultats
    results= {dataset_name: {method: [] for method in methods} for dataset_name in datasets.keys()}
    final_results = {dataset_name: {} for dataset_name in datasets.keys()}
    initial_results = {}
    full_training_results = {}
    
    for dataset_name, (X, y) in datasets.items():
        metric=METRICS[dataset_name]
        logging.info(f"\nTraitement du dataset: {dataset_name}")
        logging.info(f"\nMétrique utilisée: {metric.__name__ if hasattr(metric, '__name__') else str(metric)}")
        # Création des ensembles d'entraînement et de test
        X_train, X_pool, y_train, y_pool, X_test, y_test = split_dataset(X, y, labeled_ratio, test_ratio)
        batch_size = int(batch_ratio * (X_train.shape[0]+X_pool.shape[0]))
        
        total_samples = X.shape[0]
        labeled_percentage = (X_train.shape[0] /(X_train.shape[0]+X_pool.shape[0])) * 100
        unlabeled_percentage = (X_pool.shape[0] / (X_train.shape[0]+X_pool.shape[0])) * 100
        test_percentage = (X_test.shape[0] / total_samples) * 100

        # Affichage des tailles des ensembles
        total_samples = X.shape[0]
        logging.info(f"Taille totale du dataset {dataset_name}: {total_samples/total_samples*100:.2f}% ({total_samples}/{total_samples})")
        logging.info(f"Taille de l'ensemble de test: {test_percentage:.2f}% ({X_test.shape[0]}/{total_samples})")
        logging.info(f"Taille de l'ensemble de training: {100-test_percentage:.2f}% ({X_train.shape[0]+X_pool.shape[0]}/{total_samples})")     
        logging.info(f"Taille de l'ensemble labellisé dans le training set: {labeled_percentage:.2f}% ({X_train.shape[0]}/{X_train.shape[0]+X_pool.shape[0]})")
        logging.info(f"Taille de l'ensemble non-labellisé dans le training set: {unlabeled_percentage:.2f}% ({X_pool.shape[0]}/{X_train.shape[0]+X_pool.shape[0]})")
        logging.info(f"Nb d'itérations: {n_iterations}")
        logging.info(f"Nb de données labellisées en plus à chaque itération: {batch_size/(X_train.shape[0]+X_pool.shape[0])*100:.2f}% ({batch_size}/{X_train.shape[0]+X_pool.shape[0]})")
        
        # Évaluation initiale
        model = model_class()
        y_pred_initial = model.fit(X_train, y_train).predict(X_test)
        if metric == "PR AUC":
            #print("Classes présentes dans y_train:", np.unique(y_train, return_counts=True))
            probas = model.predict_proba(X_test)
            #print("Shape de predict_proba:", probas.shape)

            y_pred_prob_initial = model.predict_proba(X_test)[:, 1]
            initial_results[dataset_name] = calculate_pr_auc(y_test, y_pred_prob_initial)
        else:
            initial_results[dataset_name] = (
                metric(y_test, y_pred_initial, average="macro") if metric == f1_score else metric(y_test, y_pred_initial)
            )
        

        # Modèle entraîné sur l'ensemble complet des données d'entraînement
        model_full = model_class()
        model_full.fit(np.vstack((X_train, X_pool)), np.hstack((y_train, y_pool)))
        if metric == "PR AUC":
            y_pred_prob_full = model_full.predict_proba(X_test)[:, 1]
            full_training_results[dataset_name] = calculate_pr_auc(y_test, y_pred_prob_full)
        else:
            full_training_results[dataset_name] = (
                metric(y_test, model_full.predict(X_test), average="macro") if metric == f1_score else metric(y_test, model_full.predict(X_test))
            )

        # === Boucle principale d'Active Learning ===
        for method in methods:
            X_train_temp, y_train_temp = X_train.copy(), y_train.copy()
            X_pool_temp, y_pool_temp = X_pool.copy(), y_pool.copy()
            results[dataset_name][method].append(initial_results[dataset_name])
            for i in range(n_iterations):
                # Entraînement et évaluation du modèle
                model, acc = train_and_evaluate(X_train_temp, y_train_temp, X_test, y_test, model_class,metric)
                results[dataset_name][method].append(acc)
                
                # Vérification de l'existence d'échantillons non labellisés
                if len(X_pool_temp) == 0:
                    logging.info(f"\nToutes les données ont été labellisées après {i} itérations.")
                    break
                
                # Ajustement de la taille du batch si nécessaire
                actual_batch_size = min(batch_size, len(X_pool_temp))
                if actual_batch_size < batch_size:
                    logging.info(f"\nBatch réduit à {actual_batch_size} échantillons car la pool est presque vide.")
                
                # Sélection des échantillons les plus incertains
                uncertain_indices = select_uncertain_samples_general(method, model, X_pool_temp, actual_batch_size, models, X_train_temp, y_train_temp, similarity_metric='cosine')
                
                # Mise à jour des ensembles labellisés et non-labellisés
                X_train_temp, y_train_temp, X_pool_temp, y_pool_temp = update_labeled_unlabeled_sets(
                    X_train_temp, y_train_temp, X_pool_temp, y_pool_temp, uncertain_indices
                )
                
                #logging.info(f"{method} - Iteration {i+1}: {len(X_train_temp)/(X_train.shape[0]+X_pool.shape[0])*100:.2f}% ({len(X_train_temp)}/{(X_train.shape[0]+X_pool.shape[0])}) samples labeled, {metric.__name__ if hasattr(metric, '__name__') else str(metric)}: {acc:.4f} on {dataset_name}")
            
            # Final evaluation after Active Learning
            if metric == "PR AUC":
                y_pred_prob_final = model.predict_proba(X_test)[:, 1]
                pr_auc = calculate_pr_auc(y_test, y_pred_prob_final)
                final_results[dataset_name][method] = pr_auc
                #logging.info(f"Final PR AUC ({method}) on {dataset_name}: {pr_auc:.4f}")
            else:
                final_results[dataset_name][method] = (
                    metric(y_test, model.predict(X_test), average="macro") if metric == f1_score else metric(y_test, model.predict(X_test))
                )
                logging.info(f"Final {metric.__name__ if hasattr(metric, '__name__') else str(metric)} ({method}) on {dataset_name}: {final_results[dataset_name][method]:.4f}")        
        #Génération des graphiques et affichage des résultats
        result_improvements = plot_active_learning_results(dataset_name, methods, results, labeled_ratio, batch_ratio,i+1, full_training_results, final_results, initial_results,metric)
        
        #logging.info(f"Initial {metric.__name__ if hasattr(metric, '__name__') else str(metric)} on {dataset_name}: {initial_results[dataset_name]:.4f}")
        #logging.info(f"Full Training Set {metric.__name__ if hasattr(metric, '__name__') else str(metric)} on {dataset_name}: {full_training_results[dataset_name]:.4f}")

        #for method in methods:
           #logging.info(f"{metric.__name__ if hasattr(metric, '__name__') else str(metric)} Improvement ({method}) on {dataset_name}: {result_improvements[method]:.4f}")
    
    return results

In [ ]:


# Exécution de l'expérience
run_active_learning_experiment_datasets(datasets, METRICS, labeled_ratio, test_ratio, n_iterations, batch_ratio, methods, model_class, models)